# FL-IMDR Sensitivity Analysis Appendix

This notebook performs the complete one-at-a-time sensitivity experiment used in the appendix. It varies six active parameters of the numerical FL-IMDR implementation while holding all other quantities fixed. Every setting uses 100 patients, 4 heterogeneous insurers, 120 market rounds, and 30 common random seeds.

The implementation is a compact, reproducible realization of the manuscript mechanisms: stochastic patient states, constrained patient selection, insurer-specific risk learning and offers, tax-funded vouchers, target-based mandated enrollment, imitation, and clipped DP-FedAvg.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
OUT=Path('fl_imdr_sensitivity_appendix')
OUT.mkdir(exist_ok=True)
print('Output directory:', OUT.resolve())

Output directory: /mnt/data/fl_imdr_sensitivity_appendix


## Sensitivity measure

For output metric $m$ and parameter $q$, the normalized sensitivity is

$$
E_{m,q}(q)=\frac{(m(q)-m_0)/m_0}{(q-q_0)/q_0}.
$$

The analysis is one-at-a-time: only one parameter changes in each sweep, and common random seeds are used across its values.

In [2]:
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Tuple
import numpy as np
import pandas as pd
from numba import njit

N_PATIENTS = 100
N_INSURERS = 4
T = 120
D = 11
N_SEEDS = 30
LAST_WINDOW = 10

@dataclass(frozen=True)
class Config:
    tau: float = 0.10
    coverage_target: float = 0.90
    fl_interval: int = 10
    dp_sigma: float = 0.05
    imitation_weight: float = 1.0
    gap_beta: float = 3.0
    clip_C: float = 1.0
    local_lr: float = 0.055
    local_steps: int = 2
    switch_prob: float = 0.50
    force_step_pct: float = 0.05
    max_voucher: float = 450.0
    claim_scale: float = 850.0

BASELINE = Config()
SWEEPS: Dict[str, List[float]] = {
    'tau': [0.05, 0.075, 0.10, 0.125, 0.15],
    'coverage_target': [0.80, 0.85, 0.90, 0.95, 0.98],
    'fl_interval': [2, 5, 10, 20, 30],
    'dp_sigma': [0.00, 0.025, 0.05, 0.075, 0.10],
    'imitation_weight': [0.00, 0.50, 1.00, 1.50, 2.00],
    'gap_beta': [1.0, 2.0, 3.0, 4.0, 5.0],
}
PARAM_SYMBOLS = {
    'tau': r'$\\tau$',
    'coverage_target': r'$\\eth_{\\mathrm{target}}$',
    'fl_interval': r'$K$',
    'dp_sigma': r'$\\sigma_{\\mathrm{DP}}$',
    'imitation_weight': r'$\\omega_{\\mathrm{imit}}$',
    'gap_beta': r'$\\beta_g$',
}
PARAM_LABELS = {
    'tau': 'Tax/cash-back rate',
    'coverage_target': 'Coverage target',
    'fl_interval': 'Federation interval',
    'dp_sigma': 'DP noise scale',
    'imitation_weight': 'Imitation weight',
    'gap_beta': 'Coverage-gap gain',
}
METRIC_NAMES = [
    'insured_pct', 'effective_price', 'coverage', 'profit_per_insurer',
    'subsidy_per_patient', 'risk_mse', 'hhi', 'coverage_disparity',
    'goal_score', 'insurer_mse_disparity'
]
METRIC_LABELS = {
    'insured_pct': 'Insured percentage',
    'effective_price': 'Accepted effective premium',
    'coverage': 'Accepted coverage',
    'profit_per_insurer': 'Profit per insurer',
    'subsidy_per_patient': 'Subsidy per patient',
    'risk_mse': 'Risk-prediction MSE',
    'hhi': 'Market concentration (HHI)',
    'coverage_disparity': 'Risk-group coverage disparity',
    'goal_score': 'Regulator goal score',
    'insurer_mse_disparity': 'Insurer MSE disparity',
}
PRIMARY_METRICS = METRIC_NAMES[:-1]


def make_random_tape(seed: int):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, size=N_PATIENTS).astype(np.float64)
    income = np.clip(rng.normal(70_000, 25_000, size=N_PATIENTS), 10_000, 200_000).astype(np.float64)
    credit = rng.uniform(300, 850, size=N_PATIENTS).astype(np.float64)
    health = rng.beta(4, 2, size=N_PATIENTS).astype(np.float64)
    location = rng.uniform(0, 1, size=N_PATIENTS).astype(np.float64)
    wanted = rng.uniform(0.10, 0.90, size=N_PATIENTS).astype(np.float64)
    budget_frac = rng.uniform(0.07, 0.20, size=N_PATIENTS).astype(np.float64)
    return (
        age, income, credit, health, location, wanted, budget_frac,
        rng.normal(size=(T, N_PATIENTS)).astype(np.float64),
        rng.normal(size=(T, N_PATIENTS)).astype(np.float64),
        rng.normal(size=(T, N_PATIENTS)).astype(np.float64),
        rng.normal(size=(T, N_PATIENTS)).astype(np.float64),
        rng.normal(size=(T, N_PATIENTS, N_INSURERS)).astype(np.float64),
        rng.normal(size=(T, N_PATIENTS, N_INSURERS)).astype(np.float64),
        rng.random(size=(T, N_PATIENTS)).astype(np.float64),
        rng.random(size=(T, N_PATIENTS)).astype(np.float64),
        rng.random(size=(T, N_PATIENTS)).astype(np.float64),
        rng.lognormal(mean=-0.5 * 0.25**2, sigma=0.25, size=(T, N_PATIENTS)).astype(np.float64),
        rng.normal(size=(T, D)).astype(np.float64),
        rng.normal(0.0, 0.02, size=D).astype(np.float64),
        rng.integers(0, N_INSURERS, size=N_PATIENTS).astype(np.int64),
    )


def cfg_array(cfg: Config):
    return np.array([
        cfg.tau, cfg.coverage_target, float(cfg.fl_interval), cfg.dp_sigma,
        cfg.imitation_weight, cfg.gap_beta, cfg.clip_C, cfg.local_lr,
        float(cfg.local_steps), cfg.switch_prob, cfg.force_step_pct,
        cfg.max_voucher, cfg.claim_scale
    ], dtype=np.float64)


@njit(cache=True)
def sigmoid(x):
    if x > 30.0:
        return 1.0
    if x < -30.0:
        return 0.0
    return 1.0 / (1.0 + np.exp(-x))


@njit(cache=True)
def calc_risk(age, income, credit, health, noise, out):
    for i in range(N_PATIENTS):
        h = min(1.0, max(0.0, health[i]))
        c = min(1.0, max(0.0, credit[i] / 850.0))
        a = min(1.0, max(0.0, age[i] / 100.0))
        y = min(1.0, max(0.0, income[i] / 120000.0))
        base = (1.0 - 0.2 * y) * np.exp(-3.0 * h) * (0.5 * (1.0-c)**2 + 0.5 * (a*a + 0.2*a))
        r = base * (1.0 + 0.05 * noise[i])
        out[i] = min(1.0, max(0.0, r))


@njit(cache=True)
def build_features(age, income, credit, health, location, X):
    for i in range(N_PATIENTS):
        a = min(1.5, max(0.0, age[i] / 70.0))
        y = min(1.7, max(0.0, income[i] / 120000.0))
        c = min(1.0, max(0.0, credit[i] / 850.0))
        h = min(1.0, max(0.0, health[i]))
        l = min(1.0, max(0.0, location[i]))
        X[i,0]=1.0; X[i,1]=a; X[i,2]=y; X[i,3]=c; X[i,4]=h; X[i,5]=l
        X[i,6]=a*a; X[i,7]=(1.0-c)**2; X[i,8]=h*h; X[i,9]=a*h; X[i,10]=y*h


@njit(cache=True)
def dot_clip(X, w, i):
    s = 0.0
    for d in range(D):
        s += X[i,d] * w[d]
    if s < 0.0:
        return 0.0
    if s > 1.0:
        return 1.0
    return s


@njit(cache=True)
def run_one_numba(
    age0, income0, credit0, health0, location, wanted, budget_frac,
    state_income, state_health, state_credit, risk_noise,
    offer_price_noise, offer_cov_noise, accept_u, voucher_accept_u, switch_u,
    claim_noise, dp_noise, init_w, init_assign, params
):
    tau=params[0]; target=params[1]; fl_interval=int(params[2]); dp_sigma=params[3]
    imitation=params[4]; gap_beta=params[5]; clip_C=params[6]; lr=params[7]
    local_steps=int(params[8]); switch_prob=params[9]; force_step=params[10]
    max_voucher=params[11]; claim_scale=params[12]

    age=age0.copy(); income=income0.copy(); credit=credit0.copy(); health=health0.copy()
    base_income=income0.copy(); base_credit=credit0.copy(); base_health=health0.copy()
    assignment=init_assign.copy()
    insured=np.zeros(N_PATIENTS, dtype=np.uint8)
    for i in range(10): insured[i]=1
    current_price=np.zeros(N_PATIENTS); current_eff=np.zeros(N_PATIENTS); current_cov=np.zeros(N_PATIENTS)
    w_global=init_w.copy(); w_local=np.empty((N_INSURERS,D))
    for j in range(N_INSURERS):
        for d in range(D): w_local[j,d]=w_global[d]

    margin_price=np.array([-0.05,0.0,0.04,0.08])
    risk_load_price=np.array([0.25,0.45,-0.10,0.65])
    margin_cov=np.array([0.05,0.0,-0.03,0.03])
    risk_load_cov=np.array([0.15,0.25,-0.05,0.35])

    y=np.zeros(N_PATIENTS); X=np.zeros((N_PATIENTS,D)); monthly_budget=np.zeros(N_PATIENTS)
    value_per_cov=np.zeros(N_PATIENTS); pred=np.zeros((N_PATIENTS,N_INSURERS))
    price=np.zeros((N_PATIENTS,N_INSURERS)); cov=np.zeros((N_PATIENTS,N_INSURERS)); util=np.zeros((N_PATIENTS,N_INSURERS))
    best=np.zeros(N_PATIENTS,dtype=np.int64); best_u=np.zeros(N_PATIENTS)
    subsidy_paid=np.zeros(N_PATIENTS); claim=np.zeros(N_PATIENTS)
    metrics=np.zeros((T,10))

    # fixed baseline risk quartiles by rank
    calc_risk(age, income, credit, health, risk_noise[0], y)
    order=np.argsort(y); group=np.zeros(N_PATIENTS,dtype=np.int64)
    for rank in range(N_PATIENTS): group[order[rank]]=min(3,(4*rank)//N_PATIENTS)

    for t in range(T):
        if t>0:
            for i in range(N_PATIENTS):
                income[i] += 0.05*(base_income[i]-income[i]) + 0.05*base_income[i]*state_income[t,i]
                health[i] += 0.05*(base_health[i]-health[i]) + 0.05*state_health[t,i]
                credit[i] += 0.05*(base_credit[i]-credit[i]) + 0.03*base_credit[i]*state_credit[t,i]
                age[i]=min(age[i]+1.0,100.0)
                income[i]=min(200000.0,max(10000.0,income[i]))
                health[i]=min(1.0,max(0.0,health[i])); credit[i]=min(850.0,max(300.0,credit[i]))

        calc_risk(age,income,credit,health,risk_noise[t],y); build_features(age,income,credit,health,location,X)
        for i in range(N_PATIENTS):
            monthly_budget[i]=min(1000.0,max(130.0,income[i]*budget_frac[i]/36.0))
            value_per_cov[i]=monthly_budget[i]/max(wanted[i],1e-3)
            best_u[i]=-1e30; best[i]=-1
            for j in range(N_INSURERS):
                pr=dot_clip(X,w_local[j],i); pred[i,j]=pr
                p=205.0*(1.0+margin_price[j]+risk_load_price[j]*(pr-0.20))*(1.0+0.025*offer_price_noise[t,i,j])
                p=min(700.0,max(150.0,p)); price[i,j]=p
                c=0.72+margin_cov[j]-risk_load_cov[j]*(pr-0.20)+0.025*offer_cov_noise[t,i,j]
                c=min(1.0,max(0.10,c)); cov[i,j]=c
                u=value_per_cov[i]*c-p-0.10*max(p-monthly_budget[i],0.0)-0.10*value_per_cov[i]*max(wanted[i]-c,0.0)
                util[i,j]=u
                # Patient selection follows the manuscript's affordability and
                # minimum-coverage constraints.
                if p<=monthly_budget[i] and c>=wanted[i] and u>best_u[i]:
                    best_u[i]=u; best[i]=j

        for i in range(N_PATIENTS):
            can_act = insured[i]==0 or switch_u[t,i]<switch_prob
            feasible = best[i] >= 0
            pacc=sigmoid(5.0*best_u[i]/max(monthly_budget[i],1.0)) if feasible else 0.0
            accept=can_act and feasible and accept_u[t,i]<pacc
            if t<5 and insured[i]==0: accept=False
            if accept:
                j=best[i]; assignment[i]=j; insured[i]=1
                current_price[i]=price[i,j]; current_eff[i]=price[i,j]; current_cov[i]=cov[i,j]

        client_sizes=np.zeros(N_INSURERS,dtype=np.int64)
        # local gradient steps
        for j in range(N_INSURERS):
            for i in range(N_PATIENTS):
                if insured[i]==1 and assignment[i]==j: client_sizes[j]+=1
            if client_sizes[j]>0:
                for step in range(local_steps):
                    grad=np.zeros(D)
                    for i in range(N_PATIENTS):
                        if insured[i]==1 and assignment[i]==j:
                            raw=0.0
                            for d in range(D): raw += X[i,d]*w_local[j,d]
                            err=raw-y[i]
                            for d in range(D): grad[d]+=2.0*X[i,d]*err/client_sizes[j]
                    for d in range(D): w_local[j,d]-=lr*grad[d]

        if imitation>0.0:
            eta=min(0.08,0.025*imitation)
            mean_w=np.zeros(D)
            for d in range(D):
                for j in range(N_INSURERS): mean_w[d]+=w_local[j,d]/N_INSURERS
            for j in range(N_INSURERS):
                for d in range(D): w_local[j,d]+=eta*(mean_w[d]-w_local[j,d])

        preliminary_profit=np.zeros(N_INSURERS)
        for i in range(N_PATIENTS):
            subsidy_paid[i]=0.0
            if insured[i]==1:
                claim[i]=y[i]*current_cov[i]*claim_scale*claim_noise[t,i]
                preliminary_profit[assignment[i]] += current_price[i]-claim[i]
            else: claim[i]=0.0
        count_ins=0
        for i in range(N_PATIENTS): count_ins+=insured[i]
        insured_pre=count_ins/N_PATIENTS; gap=max(0.0,target-insured_pre)
        gamma_gap=(1.0+gap_beta*gap/max(target,1e-8))**2
        positive_profit=0.0
        for j in range(N_INSURERS): positive_profit+=max(preliminary_profit[j],0.0)
        pool=tau*gamma_gap*positive_profit
        uninsured_count=N_PATIENTS-count_ins
        if t>=5 and uninsured_count>0 and pool>0.0:
            voucher=min(max_voucher,pool/uninsured_count)
            for i in range(N_PATIENTS):
                if insured[i]==0:
                    j=-1; min_p=1e30
                    for jj in range(N_INSURERS):
                        if cov[i,jj]>=wanted[i] and price[i,jj]<min_p:
                            min_p=price[i,jj]; j=jj
                    if j>=0:
                        eff=max(price[i,j]-voucher,0.0)
                        uv=value_per_cov[i]*cov[i,j]-eff-0.10*max(eff-monthly_budget[i],0.0)-0.10*value_per_cov[i]*max(wanted[i]-cov[i,j],0.0)
                        if eff<=monthly_budget[i] and voucher_accept_u[t,i] < sigmoid(5.0*uv/max(monthly_budget[i],1.0)):
                            assignment[i]=j; insured[i]=1; current_price[i]=price[i,j]; current_eff[i]=eff; current_cov[i]=cov[i,j]
                            subsidy_paid[i]=min(voucher,current_price[i])

        count_ins=0
        for i in range(N_PATIENTS): count_ins+=insured[i]
        target_count=int(np.ceil(target*N_PATIENTS)); needed=max(0,target_count-count_ins)
        if t>=5 and needed>0:
            allowed=min(needed,max(1,int(np.ceil(force_step*N_PATIENTS*(1.0+gap)))))
            for rep in range(allowed):
                chosen=-1; chosen_j=0; chosen_p=1e30
                for i in range(N_PATIENTS):
                    if insured[i]==0:
                        local_j=-1; local_p=1e30
                        for j in range(N_INSURERS):
                            if cov[i,j]>=wanted[i] and price[i,j]<local_p:
                                local_p=price[i,j]; local_j=j
                        if local_j<0:
                            local_j=0; local_p=price[i,0]
                            for j in range(1,N_INSURERS):
                                if price[i,j]<local_p: local_p=price[i,j]; local_j=j
                        if local_p<chosen_p: chosen=i; chosen_j=local_j; chosen_p=local_p
                if chosen<0: break
                insured[chosen]=1; assignment[chosen]=chosen_j; current_price[chosen]=price[chosen,chosen_j]
                current_cov[chosen]=cov[chosen,chosen_j]; v=min(max_voucher,current_price[chosen])
                current_eff[chosen]=max(current_price[chosen]-v,0.0); subsidy_paid[chosen]=v

        profits=np.zeros(N_INSURERS); shares=np.zeros(N_INSURERS); local_mses=np.zeros(N_INSURERS)
        for j in range(N_INSURERS):
            n_j=0; mse_j=0.0
            for i in range(N_PATIENTS):
                if insured[i]==1 and assignment[i]==j:
                    cl=y[i]*current_cov[i]*claim_scale*claim_noise[t,i]
                    profits[j]+=current_price[i]-cl; n_j+=1
                pr=dot_clip(X,w_local[j],i); mse_j+=(pr-y[i])**2/N_PATIENTS
            shares[j]=n_j/N_PATIENTS; local_mses[j]=mse_j

        if (t+1)%fl_interval==0:
            weights=np.zeros(N_INSURERS); wsum=0.0
            for j in range(N_INSURERS): weights[j]=max(client_sizes[j],1); wsum+=weights[j]
            for j in range(N_INSURERS): weights[j]/=wsum
            avg_delta=np.zeros(D)
            for j in range(N_INSURERS):
                norm=0.0
                for d in range(D): norm+=(w_local[j,d]-w_global[d])**2
                norm=np.sqrt(norm); scale=min(1.0,clip_C/max(norm,1e-12))
                for d in range(D): avg_delta[d]+=weights[j]*(w_local[j,d]-w_global[d])*scale
            for d in range(D): w_global[d]+=avg_delta[d]+dp_sigma*dp_noise[t,d]
            for j in range(N_INSURERS):
                for d in range(D): w_local[j,d]=w_global[d]

        risk_mse=0.0
        for i in range(N_PATIENTS):
            pg=dot_clip(X,w_global,i); risk_mse+=(pg-y[i])**2/N_PATIENTS
        group_cov=np.zeros(4); group_n=np.zeros(4)
        count_ins=0; sum_eff=0.0; sum_cov=0.0; subsidy_total=0.0
        for i in range(N_PATIENTS):
            g=group[i]; group_n[g]+=1.0; group_cov[g]+=insured[i]
            if insured[i]==1: count_ins+=1; sum_eff+=current_eff[i]; sum_cov+=current_cov[i]
            subsidy_total+=subsidy_paid[i]
        for g in range(4): group_cov[g]/=max(group_n[g],1.0)
        cov_disp=np.max(group_cov)-np.min(group_cov); insured_pct=count_ins/N_PATIENTS
        eff_price=sum_eff/max(count_ins,1); acc_cov=sum_cov/max(count_ins,1)
        hhi=0.0; profit_mean=0.0
        for j in range(N_INSURERS): hhi+=shares[j]**2; profit_mean+=profits[j]/N_INSURERS
        mean_budget=0.0
        for i in range(N_PATIENTS): mean_budget+=monthly_budget[i]/N_PATIENTS
        afford=1.0-min(1.0,max(0.0,eff_price/max(mean_budget,1.0)))
        goal=0.5*insured_pct+0.3*acc_cov+0.2*afford
        max_mse=local_mses[0]; min_mse=local_mses[0]
        for j in range(1,N_INSURERS): max_mse=max(max_mse,local_mses[j]); min_mse=min(min_mse,local_mses[j])
        metrics[t,0]=insured_pct; metrics[t,1]=eff_price; metrics[t,2]=acc_cov; metrics[t,3]=profit_mean
        metrics[t,4]=subsidy_total/N_PATIENTS; metrics[t,5]=risk_mse; metrics[t,6]=hhi
        metrics[t,7]=cov_disp; metrics[t,8]=goal; metrics[t,9]=max_mse-min_mse
    return metrics


def all_settings():
    for param, values in SWEEPS.items():
        for value in values:
            cfg = replace(BASELINE, **{param: int(value) if param=='fl_interval' else float(value)})
            yield param, float(value), cfg


def convergence_month(curve):
    curve=np.asarray(curve,float); smooth=pd.Series(curve).rolling(5,min_periods=1).mean().to_numpy()
    final=float(np.mean(smooth[-10:])); initial=float(np.mean(smooth[:5]))
    if initial>final: threshold=final+0.10*(initial-final); candidates=np.where(smooth<=threshold)[0]
    else: candidates=np.where(np.abs(smooth-final)<=0.05*max(abs(final),1e-8))[0]
    return int(candidates[0]+1) if len(candidates) else T


def run_sensitivity(n_seeds=N_SEEDS):
    tapes=[make_random_tape(s) for s in range(n_seeds)]
    # warm-up compilation
    _=run_one_numba(*tapes[0],cfg_array(BASELINE))
    rows=[]; total=sum(len(v) for v in SWEEPS.values())
    for idx,(param,value,cfg) in enumerate(all_settings(),1):
        print(f'[{idx:02d}/{total}] {param}={value}',flush=True)
        p=cfg_array(cfg)
        for seed,tape in enumerate(tapes):
            arr=run_one_numba(*tape,p)
            for t in range(T):
                row={'parameter':param,'value':value,'seed':seed,'month':t+1}
                for k,name in enumerate(METRIC_NAMES): row[name]=float(arr[t,k])
                rows.append(row)
    return pd.DataFrame(rows)


def summarize_sensitivity(raw):
    per=[]
    for (param,value,seed),g in raw.groupby(['parameter','value','seed'],sort=False):
        tail=g.tail(LAST_WINDOW); row={'parameter':param,'value':value,'seed':seed}
        for metric in PRIMARY_METRICS: row[metric]=float(tail[metric].mean())
        row['convergence_month']=convergence_month(g['risk_mse'].to_numpy()); per.append(row)
    per_seed=pd.DataFrame(per)
    agg=per_seed.groupby(['parameter','value'],sort=False).agg(['mean','std','count']).reset_index()
    agg.columns=['parameter','value']+[f'{a}_{b}' for a,b in agg.columns.tolist()[2:]]
    erows=[]
    for param,values in SWEEPS.items():
        p0=float(getattr(BASELINE,param)); base=per_seed[(per_seed.parameter==param)&np.isclose(per_seed.value,p0)]
        bmeans=base[PRIMARY_METRICS].mean()
        for value in values:
            if np.isclose(value,p0): continue
            cur=per_seed[(per_seed.parameter==param)&np.isclose(per_seed.value,float(value))]
            cmeans=cur[PRIMARY_METRICS].mean(); denom=(float(value)-p0)/p0
            for metric in PRIMARY_METRICS:
                m0=float(bmeans[metric]); m=float(cmeans[metric]); e=((m-m0)/m0)/denom if abs(m0)>1e-12 else np.nan
                erows.append({'parameter':param,'value':float(value),'metric':metric,'elasticity':e,'abs_elasticity':abs(e)})
    return per_seed,agg,pd.DataFrame(erows)


def paired_tests(per_seed):
    from scipy import stats
    rows=[]
    for param,values in SWEEPS.items():
        p0=float(getattr(BASELINE,param)); base=per_seed[(per_seed.parameter==param)&np.isclose(per_seed.value,p0)].sort_values('seed')
        for metric in PRIMARY_METRICS:
            local=[]
            for value in values:
                if np.isclose(value,p0): continue
                cur=per_seed[(per_seed.parameter==param)&np.isclose(per_seed.value,float(value))].sort_values('seed')
                x=cur[metric].to_numpy(); y=base[metric].to_numpy()
                try: stat,p=stats.wilcoxon(x,y,zero_method='wilcox')
                except ValueError: stat,p=0.0,1.0
                local.append({'parameter':param,'value':float(value),'metric':metric,'W':float(stat),'p_raw':float(p)})
            pvals=np.array([r['p_raw'] for r in local]); order=np.argsort(pvals); adj=np.empty_like(pvals); running=0.0; m=len(pvals)
            for rank,ii in enumerate(order): running=max(running,min(1.0,(m-rank)*pvals[ii])); adj[ii]=running
            for r,a in zip(local,adj): r['p_holm']=float(a); rows.append(r)
    return pd.DataFrame(rows)


def build_key_summary(agg,elasticity):
    rows=[]
    for param,values in SWEEPS.items():
        p0=float(getattr(BASELINE,param)); e=elasticity[elasticity.parameter==param]
        by=e.groupby('metric')['abs_elasticity'].max().sort_values(ascending=False); metric=by.index[0]
        a=agg[agg.parameter==param]
        b=float(a[np.isclose(a.value,p0)][f'{metric}_mean'].iloc[0]); lo=float(a[np.isclose(a.value,float(values[0]))][f'{metric}_mean'].iloc[0]); hi=float(a[np.isclose(a.value,float(values[-1]))][f'{metric}_mean'].iloc[0])
        rows.append({'parameter':param,'symbol':PARAM_SYMBOLS[param],'description':PARAM_LABELS[param],'baseline':p0,'tested_min':float(values[0]),'tested_max':float(values[-1]),'most_sensitive_metric':metric,'metric_label':METRIC_LABELS[metric],'max_abs_elasticity':float(by.iloc[0]),'low_endpoint_change_pct':100*(lo-b)/b,'high_endpoint_change_pct':100*(hi-b)/b})
    return pd.DataFrame(rows)



In [3]:
raw = run_sensitivity(N_SEEDS)
per_seed, summary, elasticity = summarize_sensitivity(raw)
tests = paired_tests(per_seed)
raw.to_csv(OUT/'sensitivity_raw_timeseries.csv', index=False)
per_seed.to_csv(OUT/'sensitivity_per_seed.csv', index=False)
summary.to_csv(OUT/'sensitivity_summary.csv', index=False)
elasticity.to_csv(OUT/'sensitivity_elasticity.csv', index=False)
tests.to_csv(OUT/'sensitivity_paired_tests.csv', index=False)
summary.head()

[01/30] tau=0.05


[02/30] tau=0.075


[03/30] tau=0.1


[04/30] tau=0.125


[05/30] tau=0.15


...

[28/30] gap_beta=3.0


[29/30] gap_beta=4.0


[30/30] gap_beta=5.0


/opt/pyvenv/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,parameter,value,seed_mean,seed_std,seed_count,insured_pct_mean,insured_pct_std,insured_pct_count,effective_price_mean,effective_price_std,effective_price_count,coverage_mean,coverage_std,coverage_count,profit_per_insurer_mean,profit_per_insurer_std,profit_per_insurer_count,subsidy_per_patient_mean,subsidy_per_patient_std,subsidy_per_patient_count,risk_mse_mean,risk_mse_std,risk_mse_count,hhi_mean,hhi_std,hhi_count,coverage_disparity_mean,coverage_disparity_std,coverage_disparity_count,goal_score_mean,goal_score_std,goal_score_count,convergence_month_mean,convergence_month_std,convergence_month_count
0,tau,0.050,14.5,8.803408,30,0.976933,0.013067,30,165.141175,5.713812,30,0.776640,0.012126,30,2867.097288,136.426061,30,0.006435,0.035244,30,0.011836,0.007718,30,0.529825,0.067947,30,0.056933,0.028877,30,0.797215,0.008320,30,54.400000,38.727697,30
1,tau,0.075,14.5,8.803408,30,0.967267,0.014081,30,169.771568,3.778108,30,0.776998,0.012306,30,2846.300568,144.690565,30,0.006445,0.035300,30,0.011861,0.007726,30,0.518211,0.061347,30,0.068933,0.034676,30,0.788934,0.009849,30,57.200000,40.420548,30
2,tau,0.100,14.5,8.803408,30,0.966267,0.013204,30,166.300593,3.760820,30,0.777195,0.012225,30,2843.731985,145.225257,30,0.006445,0.035300,30,0.011853,0.007724,30,0.515951,0.062409,30,0.072933,0.031648,30,0.791118,0.009026,30,57.200000,40.420548,30
3,tau,0.125,14.5,8.803408,30,0.965600,0.013763,30,162.644003,4.000293,30,0.777188,0.012285,30,2841.635411,144.917486,30,0.006445,0.035300,30,0.011853,0.007722,30,0.515559,0.062182,30,0.075600,0.035410,30,0.793543,0.009045,30,54.366667,38.813865,30
4,tau,0.150,14.5,8.803408,30,0.965267,0.014741,30,159.145795,4.317424,30,0.777143,0.012228,30,2840.589384,145.845427,30,0.006445,0.035300,30,0.011850,0.007723,30,0.515210,0.060679,30,0.074267,0.034420,30,0.796004,0.009709,30,54.333333,38.826167,30


## Publication figures and LaTeX outputs

In [4]:
from pathlib import Path
import shutil, zipfile, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nbformat as nbf
from nbconvert.preprocessors import ExecutePreprocessor

ROOT = Path('/mnt/data')
OUT = ROOT / 'fl_imdr_sensitivity_appendix'
OUT.mkdir(exist_ok=True)

agg = pd.read_csv(OUT / 'sensitivity_summary.csv')
elasticity = pd.read_csv(OUT / 'sensitivity_elasticity.csv')
tests = pd.read_csv(OUT / 'sensitivity_paired_tests.csv')
per_seed = pd.read_csv(OUT / 'sensitivity_per_seed.csv')

SWEEPS = {
    'tau': [0.05, 0.075, 0.10, 0.125, 0.15],
    'coverage_target': [0.80, 0.85, 0.90, 0.95, 0.98],
    'fl_interval': [2, 5, 10, 20, 30],
    'dp_sigma': [0.00, 0.025, 0.05, 0.075, 0.10],
    'imitation_weight': [0.00, 0.50, 1.00, 1.50, 2.00],
    'gap_beta': [1.0, 2.0, 3.0, 4.0, 5.0],
}
BASELINES = {'tau':0.10,'coverage_target':0.90,'fl_interval':10.0,'dp_sigma':0.05,'imitation_weight':1.0,'gap_beta':3.0}
LABELS = {
    'tau': r'$\tau$',
    'coverage_target': r'$\eth_{\mathrm{target}}$',
    'fl_interval': r'$K$',
    'dp_sigma': r'$\sigma_{\mathrm{DP}}$',
    'imitation_weight': r'$\omega_{\mathrm{imit}}$',
    'gap_beta': r'$\beta_g$',
}
PLAIN = {
    'tau': 'Tax/cash-back rate',
    'coverage_target': 'Coverage target',
    'fl_interval': 'Federation interval',
    'dp_sigma': 'DP noise scale',
    'imitation_weight': 'Imitation weight',
    'gap_beta': 'Coverage-gap gain',
}
METRIC_LABELS = {
    'insured_pct': 'Insured percentage',
    'effective_price': 'Accepted effective premium',
    'coverage': 'Accepted coverage',
    'profit_per_insurer': 'Profit per insurer',
    'subsidy_per_patient': 'Subsidy per patient',
    'risk_mse': 'Risk-prediction MSE',
    'hhi': 'HHI',
    'coverage_disparity': 'Coverage disparity',
    'goal_score': 'Regulator goal score',
}
ROBUST_METRICS = ['insured_pct','effective_price','coverage','profit_per_insurer','risk_mse','hhi','coverage_disparity','goal_score']

# ---------- figures ----------
def response_plot(metric, ylabel, filename, percent=False, logy=False):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for param in SWEEPS:
        g = agg[agg.parameter == param].copy().sort_values('value')
        x = g['value'].to_numpy(float) / BASELINES[param]
        y = g[f'{metric}_mean'].to_numpy(float)
        ci = 1.96 * g[f'{metric}_std'].to_numpy(float) / np.sqrt(g[f'{metric}_count'].to_numpy(float))
        if percent:
            y = 100*y; ci = 100*ci
        ax.errorbar(x, y, yerr=ci, marker='o', linewidth=1.3, capsize=2.5, label=LABELS[param])
    ax.axvline(1.0, linestyle='--', linewidth=1.0)
    ax.set_xlabel('Parameter value / baseline value')
    ax.set_ylabel(ylabel)
    if logy: ax.set_yscale('log')
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, ncol=3, fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / f'{filename}.pdf', bbox_inches='tight')
    fig.savefig(OUT / f'{filename}.png', dpi=600, bbox_inches='tight')
    plt.close(fig)

response_plot('insured_pct', 'Insured population (%)', 'sensitivity_insured_percentage', percent=True)
response_plot('effective_price', 'Accepted effective premium', 'sensitivity_effective_premium')
response_plot('risk_mse', 'Final risk-prediction MSE', 'sensitivity_risk_mse', logy=True)
response_plot('hhi', 'Market concentration (HHI)', 'sensitivity_hhi')

# Elasticity heatmap: maximum |elasticity| over non-baseline values.
params = list(SWEEPS)
metrics = ROBUST_METRICS
mat = np.zeros((len(params), len(metrics)))
for i,p in enumerate(params):
    for j,m in enumerate(metrics):
        q = elasticity[(elasticity.parameter==p)&(elasticity.metric==m)]['abs_elasticity']
        mat[i,j] = q.max() if len(q) else np.nan
fig, ax = plt.subplots(figsize=(9.2, 4.8))
im = ax.imshow(np.log10(1.0 + mat), aspect='auto')
ax.set_xticks(np.arange(len(metrics)), [METRIC_LABELS[m] for m in metrics], rotation=38, ha='right')
ax.set_yticks(np.arange(len(params)), [LABELS[p] for p in params])
for i in range(len(params)):
    for j in range(len(metrics)):
        ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_xlabel('Output metric')
ax.set_ylabel('Varied parameter')
cb = fig.colorbar(im, ax=ax)
cb.set_label(r'$\log_{10}(1+\max|E_{m,q}|)$')
fig.tight_layout()
fig.savefig(OUT / 'sensitivity_elasticity_heatmap.pdf', bbox_inches='tight')
fig.savefig(OUT / 'sensitivity_elasticity_heatmap.png', dpi=600, bbox_inches='tight')
plt.close(fig)

# ---------- tables ----------
def fmt_value(param, v):
    if param=='fl_interval': return f'{int(round(v))}'
    if param in ('tau','coverage_target','dp_sigma'): return f'{v:.3f}'.rstrip('0').rstrip('.')
    return f'{v:.2f}'.rstrip('0').rstrip('.')

param_tex = r'''\begin{table}[t]
\centering
\caption{Parameters Included in the One-at-a-Time Sensitivity Analysis}
\label{tab:sensitivity_ranges}
\footnotesize
\renewcommand{\arraystretch}{1.08}
\begin{tabular}{lccc}
\toprule
Parameter & Baseline & Tested values & Role \\
\midrule
'''
roles = {
    'tau':'Voucher-pool scale', 'coverage_target':'Regulatory target', 'fl_interval':'FL frequency',
    'dp_sigma':'Privacy perturbation', 'imitation_weight':'Competition coupling', 'gap_beta':'Gap-response gain'
}
for p, vals in SWEEPS.items():
    values=', '.join(fmt_value(p,float(v)) for v in vals)
    param_tex += f"{LABELS[p]} & {fmt_value(p,BASELINES[p])} & {values} & {roles[p]} " + r"\\" + "\n"
param_tex += r'''\bottomrule
\end{tabular}
\end{table}
'''
(OUT/'sensitivity_parameter_ranges.tex').write_text(param_tex)

# Key summary excluding subsidy elasticity because its baseline is numerically close to zero.
key_rows=[]
for p in params:
    e=elasticity[(elasticity.parameter==p)&elasticity.metric.isin(ROBUST_METRICS)]
    by=e.groupby('metric')['abs_elasticity'].max().sort_values(ascending=False)
    metric=by.index[0]
    g=agg[agg.parameter==p]
    p0=BASELINES[p]
    b=float(g[np.isclose(g.value,p0)][f'{metric}_mean'].iloc[0])
    lo=float(g[np.isclose(g.value,float(SWEEPS[p][0]))][f'{metric}_mean'].iloc[0])
    hi=float(g[np.isclose(g.value,float(SWEEPS[p][-1]))][f'{metric}_mean'].iloc[0])
    key_rows.append({'parameter':p,'metric':metric,'elasticity':float(by.iloc[0]),'baseline_metric':b,'low':lo,'high':hi})
key_df=pd.DataFrame(key_rows)
key_df.to_csv(OUT/'sensitivity_key_summary_robust.csv',index=False)

key_tex = r'''\begin{table*}[t]
\centering
\caption{Dominant Responses in the Sensitivity Analysis}
\label{tab:sensitivity_key}
\footnotesize
\renewcommand{\arraystretch}{1.10}
\begin{tabular}{lcccc}
\toprule
Parameter & Principal response & $\max |E_{m,q}|$ & Low-end result & High-end result \\
\midrule
'''
for _,r in key_df.iterrows():
    p=r.parameter; m=r.metric
    def metric_fmt(v):
        if m=='insured_pct' or m=='coverage' or m=='coverage_disparity' or m=='goal_score': return f'{100*v:.2f}\\%'
        if m=='effective_price' or m=='profit_per_insurer': return f'{v:.2f}'
        return f'{v:.4f}'
    key_tex += f"{LABELS[p]} & {METRIC_LABELS[m]} & {r.elasticity:.2f} & {metric_fmt(r.low)} & {metric_fmt(r.high)} " + r"\\" + "\n"
key_tex += r'''\bottomrule
\end{tabular}
\end{table*}
'''
(OUT/'sensitivity_key_table.tex').write_text(key_tex)

# Detailed results in two tables.
def detail_table(group_params, label, caption):
    tex = rf'''\begin{{table*}}[t]
\centering
\caption{{{caption}}}
\label{{{label}}}
\scriptsize
\renewcommand{{\arraystretch}}{{1.04}}
\setlength{{\tabcolsep}}{{3.2pt}}
\begin{{tabular}}{{llcccccc}}
\toprule
Parameter & Value & Insured (\%) & Eff. premium & Profit/insurer & Risk MSE & HHI & $\Delta_{{\mathrm{{cov}}}}$ \\
\midrule
'''
    for p in group_params:
        g=agg[agg.parameter==p].sort_values('value')
        first=True
        for _,r in g.iterrows():
            pname=LABELS[p] if first else ''
            tex += (f"{pname} & {fmt_value(p,float(r.value))} & "
                    f"${100*r.insured_pct_mean:.2f} \\pm {100*r.insured_pct_std:.2f}$ & "
                    f"${r.effective_price_mean:.2f} \\pm {r.effective_price_std:.2f}$ & "
                    f"${r.profit_per_insurer_mean:.1f} \\pm {r.profit_per_insurer_std:.1f}$ & "
                    f"${r.risk_mse_mean:.4f} \\pm {r.risk_mse_std:.4f}$ & "
                    f"${r.hhi_mean:.3f} \\pm {r.hhi_std:.3f}$ & "
                    f"${r.coverage_disparity_mean:.3f} \\pm {r.coverage_disparity_std:.3f}$ " + r"\\" + "\n")
            first=False
        tex += r'\addlinespace'+'\n'
    tex += r'''\bottomrule
\end{tabular}
\end{table*}
'''
    return tex

detail_a=detail_table(['tau','coverage_target','gap_beta'],'tab:sensitivity_regulator','Sensitivity of Regulatory and Market-Response Parameters (30 Seeds)')
detail_b=detail_table(['fl_interval','dp_sigma','imitation_weight'],'tab:sensitivity_learning','Sensitivity of Learning and Coordination Parameters (30 Seeds)')
(OUT/'sensitivity_detailed_tables.tex').write_text(detail_a+'\n'+detail_b)

# ---------- appendix text ----------
appendix = r'''\section{Sensitivity Analysis over Key Parameters}
\label{app:sensitivity}

A one-at-a-time sensitivity analysis was conducted for the six active parameters listed in Table~\ref{tab:sensitivity_ranges}. Each configuration was simulated for $120$ market rounds using $30$ common random seeds, while all parameters other than the parameter under examination were fixed at their baseline values. The reported values are means over the final ten rounds and are shown as mean $\pm$ standard deviation. This experiment evaluates local robustness of the simulated FL-IMDR market and is not intended as real-world external validation.

For an output metric $m$ and parameter $q$ with baseline values $m_0$ and $q_0$, the normalized sensitivity is summarized using
\begin{equation}
\label{eq:sensitivity_elasticity}
E_{m,q}(q)=
\frac{\bigl(m(q)-m_0\bigr)/m_0}
     {\bigl(q-q_0\bigr)/q_0}.
\end{equation}
The quantity $|E_{m,q}|<1$ indicates that the relative output change is smaller than the relative parameter perturbation, whereas $|E_{m,q}|>1$ identifies a comparatively sensitive response. The numerical implementation uses the gap-amplification parameter $\beta_g$ through
\begin{equation}
\gamma_t(\beta_g)=
\left(1+\beta_g
\frac{\max(0,\eth_{\mathrm{target}}-\eth_t)}
     {\eth_{\mathrm{target}}}\right)^2,
\end{equation}
with baseline $\beta_g=3$.

\input{Appendix/sensitivity_parameter_ranges.tex}

Figures~\ref{fig:sensitivity_insured}--\ref{fig:sensitivity_hhi} show that the market-level outcomes are comparatively stable under changes in $\tau$, $\omega_{\mathrm{imit}}$, and $\beta_g$, while the learning outcomes are more sensitive to $K$ and $\sigma_{\mathrm{DP}}$. Increasing $\eth_{\mathrm{target}}$ from $0.80$ to $0.98$ increased the steady-state insured population from $96.17\%$ to $99.08\%$ and reduced the risk-group coverage disparity from $0.079$ to $0.032$. The effective premium simultaneously decreased from $170.81$ to $158.68$, because the stronger target activated more regulatory assistance.

The strongest learning sensitivity was observed for the privacy perturbation. Increasing $\sigma_{\mathrm{DP}}$ from $0$ to $0.10$ increased the final risk MSE from $0.0013\pm0.0007$ to $0.0362\pm0.0305$ and increased HHI from $0.487\pm0.067$ to $0.533\pm0.068$. Both changes were significant relative to the baseline $\sigma_{\mathrm{DP}}=0.05$ after paired Wilcoxon testing and Holm correction for the corresponding comparisons. The federation interval produced a non-monotone response: $K=2$ repeatedly injected DP noise and gave MSE $0.0240\pm0.0160$, whereas $K=20$ limited information sharing and gave MSE $0.0275\pm0.0309$, compared with $0.0119\pm0.0111$ at $K=10$. This indicates that neither excessively frequent nor excessively sparse federation is uniformly preferable in the dynamic market.

The tax rate and gap-response gain mainly affected affordability rather than predictive accuracy. Increasing $\tau$ from $0.05$ to $0.15$ reduced the effective premium from $165.14$ to $159.15$, while increasing $\beta_g$ from $1$ to $5$ reduced it from $163.63$ to $156.16$. In contrast, varying $\omega_{\mathrm{imit}}$ over $[0,2]$ changed the principal metrics only slightly, indicating that the reported behavior is not driven by a narrowly tuned imitation coefficient. Table~\ref{tab:sensitivity_key} and Fig.~\ref{fig:sensitivity_elasticity} summarize the dominant responses, while Tables~\ref{tab:sensitivity_regulator} and~\ref{tab:sensitivity_learning} report all tested configurations.

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{Figures/sensitivity_insured_percentage.pdf}
\caption{Steady-state insured percentage under one-at-a-time parameter variations. Error bars denote $95\%$ confidence intervals over $30$ common seeds, and the dashed line marks the baseline parameter value.}
\label{fig:sensitivity_insured}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{Figures/sensitivity_effective_premium.pdf}
\caption{Sensitivity of the accepted effective premium. Stronger regulatory targets, taxation, and coverage-gap response generally reduce the patient-paid premium.}
\label{fig:sensitivity_premium}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{Figures/sensitivity_risk_mse.pdf}
\caption{Sensitivity of final risk-prediction MSE. The logarithmic scale highlights the stronger dependence on the DP noise level and federation interval.}
\label{fig:sensitivity_mse}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\columnwidth]{Figures/sensitivity_hhi.pdf}
\caption{Sensitivity of market concentration measured by the Herfindahl--Hirschman index.}
\label{fig:sensitivity_hhi}
\end{figure}

\begin{figure*}[t]
\centering
\includegraphics[width=0.92\textwidth]{Figures/sensitivity_elasticity_heatmap.pdf}
\caption{Maximum absolute normalized elasticity in~\eqref{eq:sensitivity_elasticity} over the tested non-baseline values. The cell labels report $\max |E_{m,q}|$; the displayed intensity uses $\log_{10}(1+\max|E_{m,q}|)$ only for visual scaling.}
\label{fig:sensitivity_elasticity}
\end{figure*}

\input{Appendix/sensitivity_key_table.tex}
\input{Appendix/sensitivity_detailed_tables.tex}
'''
(OUT/'appendix_sensitivity_analysis.tex').write_text(appendix)

# ---------- README ----------
readme = '''FL-IMDR sensitivity appendix package

Experiment:
- 100 patients, 4 insurers, 120 monthly rounds
- 30 common random seeds per setting
- one-at-a-time sweeps of tau, coverage_target, fl_interval, dp_sigma,
  imitation_weight, and gap_beta
- final metrics averaged over the last 10 rounds
- paired Wilcoxon comparisons against each parameter's baseline with Holm correction

The notebook is self-contained and reproduces the simulation, CSV files,
figures, LaTeX tables, and appendix text. It uses a compact vectorized/Numba
implementation of the manuscript's patient dynamics, decentralized offer
selection, tax-funded vouchers, mandated target response, insurer risk
learning, imitation, and clipped noisy FedAvg. It is a controlled appendix
experiment and should be described as empirical sensitivity within the
simulated market, not as external validation.
'''
(OUT/'README.txt').write_text(readme)



859

## Main findings

- The insured percentage and economic outcomes are relatively stable for moderate changes in the tax rate, imitation weight, and gap-response gain.
- The target coverage directly changes achieved coverage and risk-group disparity.
- Risk-prediction accuracy is most sensitive to the DP noise scale and the federation interval.
- The federation interval has a non-monotone effect because very frequent aggregation repeatedly injects DP noise, whereas sparse aggregation delays cross-insurer information sharing.